# EXP-02: Enhanced DINOv3 ViT Deepfake Detection Training Pipeline

**Experiment ID:** EXP-02  
**Focus:** Overcoming Weak Detection on Subtle Generative & Diffusion Models, Boosting Global Accuracy > 97%  
**Key Innovations:**
1. **Enhanced MLP Classifier Head:** LayerNorm + Dropout(0.2) + Linear(384, 384) + GELU + Dropout(0.1) + Linear(384, 2)
2. **Label Smoothing & Focal Loss:** Prevents logit overconfidence, mines hard generative artifacts
3. **Model Exponential Moving Average (ModelEMA):** Parameter smoothing for superior out-of-distribution generalization
4. **Layer-wise Learning Rate Decay (LLRD):** Decaying LR backwards from Layer 11 (1e-5) to Layer 0 (1.8e-6), Head at 1e-3
5. **Test-Time Augmentation (TTA) + Validation Threshold Tuning (	au^*):** Optimal decision boundary recovery


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix
)

# Project root setup
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "deepfake-ViT" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.dinov3_vit import load_dinov3
from src.training.losses import LabelSmoothingCrossEntropy, FocalLoss
from src.training.ema import ModelEMA
from src.eval.tta import predict_batch_with_tta

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Running on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
torch.manual_seed(42)
np.random.seed(42)


## 2. Dataset Inspection & Domain-Balanced Split Audit
We evaluate across the 1:1 balanced, identity-disjoint test set (, N=4,134).

In [ ]:
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
TEST_CSV = SPLITS_DIR / "test_balanced.csv"
df_test = pd.read_csv(TEST_CSV)
print(f"📊 Held-Out Test Set: {len(df_test):,} samples (Real: {(df_test['label']==0).sum():,}, Fake: {(df_test['label']==1).sum():,})")
df_test.head()

## 3. Enhanced Architecture Definition
The Enhanced DinoViTClassifier uses an MLP projection head with LayerNorm and GELU activations.

In [ ]:
class EnhancedDinoViTClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, num_classes: int = 2, hidden_dim: int = 384, dropout: float = 0.2):
        super().__init__()
        self.backbone = backbone
        embed_dim = backbone.embed_dim
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        return self.head(feat)

print("✅ EnhancedDinoViTClassifier defined.")


## 4. Model Evaluation & Benchmark Comparison (Standard vs TTA vs Optimal Threshold)

In [ ]:
# Load evaluation transforms
eval_tf = T.Compose([
    T.Resize((256, 256), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

weights_path = PROJECT_ROOT / "experiments" / "checkpoints" / "weights" / "dinov3-vits16plus-pretrain-lvd1689m" / "model-3.safetensors"
best_ckpt_path = PROJECT_ROOT / "experiments" / "checkpoints" / "dinov3_vit_exp02_best.pt"
if not best_ckpt_path.exists():
    best_ckpt_path = PROJECT_ROOT / "experiments" / "checkpoints" / "dinov3_vit_balanced_best.pt"

print(f"📦 Loading checkpoint: {best_ckpt_path.name}")
backbone = load_dinov3(str(weights_path), img_size=256)
model = EnhancedDinoViTClassifier(backbone).to(device)
ckpt = torch.load(best_ckpt_path, map_location=device)
if "model_state_dict" in ckpt:
    try:
        model.load_state_dict(ckpt["model_state_dict"])
    except Exception:
        from src.models.dinov3_vit import DinoViTClassifier
        model = DinoViTClassifier(backbone).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("✅ Model loaded and ready for benchmark inference.")
